# 10 — Partition and leakage audit (Table 1)
Read-only. Verifies the 76-slide split (38/13/13/12), that every leakage group and every resolved biological sample ID lies in exactly one split, the consistency with the older 78-slide file and the mapping table, and the model-ready package; then prints the split-summary tables behind Table 1. Passing the group check does **not** prove that slides with unresolved identity come from different patients.

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import sys, pandas as pd
from pathlib import Path
sys.path.insert(0, '/content/spatial-sasp-gat/src')
from sasp_gat.partition import partition_audit, sha_file
B = Path(BASE)
F3, F78, FM, PKL = B/'FINAL_split_assignment_v3_SASP54_2026-09-26.csv', B/'FINAL_split_assignment_2026-09-26.csv', B/'FINAL_biological_sample_mapping_v2_2026-09-26.csv', B/'MODEL_READY_DATASET_v1_2026-09-26.pkl'
for p in (F3, F78, FM, PKL): print(p.name, sha_file(p))
R = partition_audit(F3, F78, FM, PKL)
print(R['n_checks'], 'checks passed')
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 70)
for k in ['T1', 'T2', 'missing_cancers', 'T3', 'T4', 'excluded']: print('\n', k); print(R[k].to_string(index=False))
OUT = B / 'repro_outputs_2026-10-24'; OUT.mkdir(exist_ok=True)
for k in ['T1', 'T2', 'T3', 'T4', 'slides']: R[k].to_csv(OUT / f'partition_{k}.csv', index=False)